In [2]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [3]:
ruta_OF = os.path.join("res\\raw", "CPRManufacturingOrder.csv")
print(ruta_OF)

res\raw\CPRManufacturingOrder.csv


In [4]:
columns_OF = ["IDManufacturingOrder", "CodCompany",	"CodManufacturingOrder", "IDArticle", "IDStructure", "Description", "TimeUnitDefault",	"IDStructureType",	"Notes", "Quantity", "IDWareHouse", "PlannedEndDate", "PlannedStartDate", "PlannedEndMRPDate", "ManualEndDate", "Priority", "IDMOSituation", "Color", "Group", "InternalCode", "IDRoute", "AcceptedDate", "RealEndDate", "IDProject", "IDProTask", "IDProMaterial", "AutomaticCodification", "ManualPStartDate", "ManualPEndDate", "IDSite", "Historic", "RowTimestamp", "PlanningDirection", "LastModifiedBy", "QuantityBaseFormula", "IDUnitBaseFormula", "IDArticleVersion", "CreationTimestamp", "RowCreatedBy", "IDMaintenanceOrder", "IDOrderMaterialPrevision",	"IDLaunchedMRP", "AutoBatchAssignSalesOrders"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_OF = pd.read_csv(ruta_OF, sep=";", header=None, names=columns_OF)

C:\Users\jaume\AppData\Local\Temp\ipykernel_24848\1230745797.py:6: DtypeWarning: Columns (0: Group) have mixed types. Specify dtype option on import or set low_memory=False.
  df_OF = pd.read_csv(ruta_OF, sep=";", header=None, names=columns_OF)


In [5]:
display(df_OF.head(1))

,IDManufacturingOrder,CodCompany,CodManufacturingOrder,IDArticle,IDStructure,Description,TimeUnitDefault,IDStructureType,Notes,Quantity,IDWareHouse,PlannedEndDate,PlannedStartDate,PlannedEndMRPDate,ManualEndDate,Priority,IDMOSituation,Color,Group,InternalCode,IDRoute,AcceptedDate,RealEndDate,IDProject,IDProTask,IDProMaterial,AutomaticCodification,ManualPStartDate,ManualPEndDate,IDSite,Historic,RowTimestamp,PlanningDirection,LastModifiedBy,QuantityBaseFormula,IDUnitBaseFormula,IDArticleVersion,CreationTimestamp,RowCreatedBy,IDMaintenanceOrder,IDOrderMaterialPrevision,IDLaunchedMRP,AutoBatchAssignSalesOrders
0,000744d2-5b9e-40be-9126-9a8a66954f33,100,OF2502249,921ac292-f882-47f8-ae0b-ce82c4f1a3c8,9e856d02-3287-4b00-b968-3e6625ce343a,CONDUCTEUR 103127410,2,d3bd7234-a0af-4c35-b658-dcd8ac943c67,<p>A1</p>,7.0,8dc40f3a-f875-4dbf-8844-63282db4196f,2025-04-07 23:59:59.000,2025-04-07 17:44:00.000,2025-04-08 00:00:00.000,2025-04-08 00:00:00.000,0,f57597bd-79b3-4a87-9f62-fdd434de7804,-4511865.0,NaN,CV251138/0036/CORXA1X0024/1,cf345109-6d07-423f-9a2e-bf48b8c569c8,2025-03-05 00:00:00.000,2025-04-17 00:00:00.000,NaN,NaN,NaN,1,2025-04-07 17:44:00.000,2025-04-07 23:59:59.000,5565bc54-9d9c-4dc7-91e7-bb5e206c9bd7,0,2026-06-15 18:32:07.553,1,JMAU,7.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb,NaN,2025-03-05 10:50:29.903,ICRE,NaN,NaN,622d3b9e-0ddb-4f25-9cee-4a75a8043531,0


In [6]:
display(df_OF.shape)

(20659, 43)

# Limpieza de OF.
Hay muchas ordenes de fabricación, para que el modelo tenga sentido debó moldearlo con los tiempos reales totales, es decir, no me sirve trabajar con OF que no estan terminadas, o que tienen imputaciones a medias. 

Esto lo haré de dos formas:

1. Descartando todas las OF las cuales su estado (IDMOSituation) sea "Creada". Ya que OF con este estado nunca han recivido imputaciones de tiempo.

2. Descartando todas las OF las cuales su fecha de finalización (RealEndDate) sea antes del 01-08-2026. 

## 1. Descartar OF por su estado
Las fases que su estado sea creadas debemos eliminarlas, el estado "Creada" puede tener dos ID:    
    - ID 100: 7776bcc3-98be-48f1-913a-30bbb40ef0bb   
    - ID 200: 7776bcc3-98be-48f1-913a-30bbb40ef0bb200   

In [7]:
df_OF_Creada = df_OF[df_OF['IDMOSituation'].isin(["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"])]
print(f"Total de OFs creadas: {df_OF_Creada.shape[0]}")

df_OF = df_OF[~df_OF['IDMOSituation'].isin(["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"])]
print(f"Total de OFs tras filtrar por situación: {df_OF.shape[0]}")

Total de OFs creadas: 1
Total de OFs tras filtrar por situación: 20658


## 2. Decartar OF por su fecha de finalización
Aquellas OF con fecha de finalización posterior al 01/08/2026 las descartaremos ya que la totalidad de sus fases no estarán terminadas. 

Esto lo hare con la columna `ManualEndDate`. 

In [8]:
df_OF["ManualEndDate"] = pd.to_datetime(df_OF["ManualEndDate"])
print(f"Total OFs {df_OF.shape[0]}")

df_OF_validas = df_OF[df_OF["ManualEndDate"] < "2026-08-01"]
print(f"Total OFs válidas {df_OF_validas.shape[0]}")
print(f"Total de OFs fuera de fecha {df_OF.shape[0] - df_OF_validas.shape[0]}")

Total OFs 20658
Total OFs válidas 19575
Total de OFs fuera de fecha 1083


Guardamos el `df_OF_validas` en el repositorio

In [9]:
df_OF_validas.to_parquet("res//procesed/df_OF_validas.parquet")